In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

X = train.drop(columns=["diagnosed_diabetes"])
y = train["diagnosed_diabetes"]
X_test = test.copy()

categorical_cols = [
    'gender', 'ethnicity', 'education_level',
    'income_level', 'smoking_status', 'employment_status'
]

# Convert to category dtype
for col in categorical_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


/Users/jose/Documents/2025 Kaggle Playground Competitions/playground-series-s5e12/playground-series-s5e12/kaggle_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def objective(trial):

    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "seed": 42,
        "num_threads": -1,

        # ---- Optimized Search Space (faster + stable) ----
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 150),
        "max_depth": trial.suggest_int("max_depth", -1, 8),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.65, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 8),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 10.0),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0)
    }

    oof_preds = np.zeros(len(X))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        train_set = lgb.Dataset(X_train, y_train, categorical_feature=categorical_cols)
        valid_set = lgb.Dataset(X_valid, y_valid, categorical_feature=categorical_cols)

        model = lgb.train(
            params,
            train_set,
            valid_sets=[valid_set],
            num_boost_round=1500,                      # 🚀 Faster tuning
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
            ]
        )

        preds = model.predict(X_valid, num_iteration=model.best_iteration)
        oof_preds[valid_idx] = preds

        # ---- Fast Pruning ----
        fold_auc = roc_auc_score(y_valid, preds)
        trial.report(fold_auc, fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return roc_auc_score(y, oof_preds)

In [3]:
# ---- Aggressive Pruner (big time-saver) ----
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(objective, n_trials=150, show_progress_bar=True)

print("Best AUC:", study.best_value)
print("Best Parameters:", study.best_params)

[I 2025-12-12 14:38:04,666] A new study created in memory with name: no-name-d747912a-a1e6-4a21-ae54-8d05b368f68c
Best trial: 0. Best value: 0.727301:   1%|          | 1/150 [01:10<2:55:17, 70.59s/it]

[I 2025-12-12 14:39:15,265] Trial 0 finished with value: 0.7273011083639856 and parameters: {'learning_rate': 0.0508753768266444, 'num_leaves': 52, 'max_depth': 0, 'min_data_in_leaf': 12, 'feature_fraction': 0.8156639966770716, 'bagging_fraction': 0.6948515891691253, 'bagging_freq': 4, 'lambda_l1': 0.24098636802953766, 'lambda_l2': 7.751481000163238, 'min_split_gain': 0.6037603614125864}. Best is trial 0 with value: 0.7273011083639856.


Best trial: 0. Best value: 0.727301:   1%|▏         | 2/150 [02:05<2:31:58, 61.61s/it]

[I 2025-12-12 14:40:10,600] Trial 1 finished with value: 0.7052288912330625 and parameters: {'learning_rate': 0.011201520015049721, 'num_leaves': 55, 'max_depth': 2, 'min_data_in_leaf': 166, 'feature_fraction': 0.6867277277584559, 'bagging_fraction': 0.7791416497252104, 'bagging_freq': 8, 'lambda_l1': 3.9687036367587627, 'lambda_l2': 3.03993054545833, 'min_split_gain': 0.1845081972027679}. Best is trial 0 with value: 0.7273011083639856.


Best trial: 0. Best value: 0.727301:   2%|▏         | 3/150 [02:58<2:20:42, 57.43s/it]

...

[I 2025-12-17 17:20:56,081] Trial 147 finished with value: 0.7295648973850044 and parameters: {'learning_rate': 0.08444027047215787, 'num_leaves': 77, 'max_depth': 5, 'min_data_in_leaf': 124, 'feature_fraction': 0.8755963179275023, 'bagging_fraction': 0.9566084506526098, 'bagging_freq': 1, 'lambda_l1': 8.141507407841946, 'lambda_l2': 1.8313035711684194, 'min_split_gain': 0.8256608332975193}. Best is trial 135 with value: 0.7297092320132457.


Best trial: 135. Best value: 0.729709:  99%|█████████▉| 149/150 [123:41:22<22:53, 1373.26s/it]

[I 2025-12-17 18:19:26,898] Trial 148 finished with value: 0.7292288037673728 and parameters: {'learning_rate': 0.07410191281327064, 'num_leaves': 75, 'max_depth': 4, 'min_data_in_leaf': 120, 'feature_fraction': 0.8697801182957495, 'bagging_fraction': 0.9233249426686032, 'bagging_freq': 1, 'lambda_l1': 8.152059237733647, 'lambda_l2': 1.7322790898925227, 'min_split_gain': 0.8378300365950659}. Best is trial 135 with value: 0.7297092320132457.


Best trial: 135. Best value: 0.729709: 100%|██████████| 150/150 [123:52:46<00:00, 2973.11s/it]

[I 2025-12-17 18:30:51,266] Trial 149 pruned. 
Best AUC: 0.7297092320132457
Best Parameters: {'learning_rate': 0.09401414688928351, 'num_leaves': 70, 'max_depth': 5, 'min_data_in_leaf': 133, 'feature_fraction': 0.8786512018414341, 'bagging_fraction': 0.9394839783643817, 'bagging_freq': 1, 'lambda_l1': 8.051867827659876, 'lambda_l2': 2.177536366098622, 'min_split_gain': 0.75421095792875}


In [4]:
# -------------------------------------
# Train final model with best parameters
# -------------------------------------
best_params = study.best_params
best_params.update({
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "seed": 42,
    "num_threads": -1
})

final_train_set = lgb.Dataset(X, y)

final_model = lgb.train(
    best_params,
    final_train_set,
    num_boost_round=3000    # reasonable large number
)

test_probs = final_model.predict(X_test)

In [7]:
# Create submission
submission = pd.DataFrame({
    "id": test["id"],
    "diagnosed_diabetes": test_probs
})
submission.to_csv("../Submissions/LightGBM_Updated.csv", index=False)